<a href="https://colab.research.google.com/github/nilesh-Joshi/Nilesh-Joshi/blob/main/Mini_Scientific_Discovery_Engine_Nilesh_Joshi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Mini Scientific Discovery Engine – Excel Dataset Version
 - Uses local Lipophilicity.xlsx (experimental logP)
 - GNN: trained on real lipophilicity data
 - Generator: assembles substituted benzenes from fragments
 - RL (DQN): discovers optimal substituent combinations
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch
import random
import numpy as np
from collections import deque
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# RDKit for chemistry
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


# 1. Fragment library & builder

SCAFFOLD = Chem.MolFromSmiles('c1ccccc1')         # benzene
FRAGMENTS = ['', 'C', 'O', 'N', 'Cl']              # none, -CH3, -OH, -NH2, -Cl
POSITIONS = [0, 1, 2, 3]                           # four substitution sites
ACTION_SPACE = len(FRAGMENTS)                      # 5 actions per position

def build_molecule(combination):
    """
    combination: list of length 4, each element index into FRAGMENTS.
    Build a valid benzene derivative by attaching fragments to ring carbons.
    Returns SMILES string or None if invalid.
    """
    frag_smiles = [FRAGMENTS[i] for i in combination]
    ring = "c1"
    for frag in frag_smiles:
        if frag == '':
            ring += "c"
        else:
            ring += f"c({frag})"
    ring += "c1"
    smi = ring
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
        return Chem.MolToSmiles(mol)
    except:
        return None


# 2. Convert SMILES to graph (extended atomic features)

MAX_ATOMIC_NUM = 100   # covers all common organic elements
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    atom_features = []
    for atom in mol.GetAtoms():
        atomic_num = atom.GetAtomicNum()
        if atomic_num < 1 or atomic_num > MAX_ATOMIC_NUM:
            return None   # skip molecules with uncommon atoms
        # one-hot atomic number (1..MAX_ATOMIC_NUM) + degree
        feat = [0.0] * (MAX_ATOMIC_NUM + 1)
        feat[atomic_num - 1] = 1.0   # index 0 for atomic num 1, etc.
        feat[-1] = float(atom.GetDegree())
        atom_features.append(feat)
    x = torch.tensor(atom_features, dtype=torch.float)

    row, col = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        row += [i, j]
        col += [j, i]
    edge_index = torch.tensor([row, col], dtype=torch.long)

    return Data(x=x, edge_index=edge_index)


# 3. GNN property predictor

INPUT_DIM = MAX_ATOMIC_NUM + 1   # one-hot length + degree
HIDDEN_DIM = 64
class GNNPredictor(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.fc(x).squeeze(-1)


# 4. Load dataset from Excel file

def load_lipophilicity_excel(filepath='/content/Lipophilicity.csv', max_rows=None):
    """
    Load experimental logP data from an Excel or CSV file.
    Expected columns: one with SMILES strings, one with logP values.
    Automatically detects column names (smiles/SMILES, exp/logP/measured).
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")

    print(f"Loading data from {filepath} ...")

    # Try Excel first, then CSV
    if filepath.endswith('.csv'):
        df = pd.read_csv(filepath)
    else:
        df = pd.read_excel(filepath)

    print(f"File contains {len(df)} rows")

    # Detect column names
    smiles_col = None
    exp_col = None

    for col in df.columns:
        col_lower = col.lower()
        if col_lower in ['smiles', 'smile', 'canonical_smiles', 'smiles_stand']:
            smiles_col = col
        elif col_lower in ['exp', 'logp', 'log_p', 'measured', 'experimental', 'logp_exp', 'logp exp']:
            exp_col = col

    if smiles_col is None:
        # Fallback: assume first column is SMILES
        smiles_col = df.columns[0]
        print(f"Assuming column '{smiles_col}' contains SMILES")
    if exp_col is None:
        # Fallback: assume second column is logP
        exp_col = df.columns[1]
        print(f"Assuming column '{exp_col}' contains logP values")

    print(f"Using: SMILES = '{smiles_col}', logP = '{exp_col}'")

    graphs, logp_vals = [], []
    skipped = 0
    for _, row in df.iterrows():
        try:
            smi = str(row[smiles_col]).strip()
            logp = float(row[exp_col])

            graph = smiles_to_graph(smi)
            if graph is not None:
                graphs.append(graph)
                logp_vals.append(logp)
            else:
                skipped += 1
        except (ValueError, KeyError) as e:
            skipped += 1
            continue

        if max_rows and len(graphs) >= max_rows:
            break

    print(f"Loaded {len(graphs)} valid molecules (skipped {skipped} invalid).")
    if len(graphs) == 0:
        raise ValueError("No valid molecules found in the file. Check SMILES and logP columns.")
    return graphs, torch.tensor(logp_vals, dtype=torch.float)


# 5. Q‑Network & DQN Agent

class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions=ACTION_SPACE * len(POSITIONS)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions)
        )

    def forward(self, state):
        return self.net(state)

class SimpleDQN:
    def __init__(self, gnn, state_dim, n_actions, lr=1e-3, target_norm=None):
        """
        target_norm: tuple (mean, std) to denormalize GNN predictions
        """
        self.gnn = gnn
        self.target_mean, self.target_std = target_norm if target_norm else (0.0, 1.0)
        self.q_net = QNetwork(state_dim, n_actions)
        self.target_net = QNetwork(state_dim, n_actions)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.memory = deque(maxlen=2000)
        self.n_actions = n_actions
        self.epsilon = 0.3
        self.gamma = 0.95
        self.batch_size = 32

    def get_state_embedding(self, combination):
        """Use GNN to embed the current molecule (graph)."""
        smi = build_molecule(combination)
        if smi is None:
            return None
        graph = smiles_to_graph(smi)
        if graph is None:
            return None
        with torch.no_grad():
            x, edge_index = graph.x, graph.edge_index
            batch = torch.zeros(x.size(0), dtype=torch.long)
            x = F.relu(self.gnn.conv1(x, edge_index))
            x = F.relu(self.gnn.conv2(x, edge_index))
            emb = global_mean_pool(x, batch)
        return emb.squeeze(0)   # [hidden_dim]

    def predict_logp(self, combination):
        """Return predicted logP (denormalized) for a combination."""
        smi = build_molecule(combination)
        if smi is None:
            return None
        graph = smiles_to_graph(smi)
        if graph is None:
            return None
        with torch.no_grad():
            batch = Batch.from_data_list([graph])
            pred_norm = self.gnn(batch).item()
            return pred_norm * self.target_std + self.target_mean

    def choose_action(self, state_emb):
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        q_values = self.q_net(state_emb)
        return torch.argmax(q_values).item()

    def store_transition(self, s, a, r, s_next, done):
        self.memory.append((s, a, r, s_next, done))

    def update(self):
        if len(self.memory) < self.batch_size:
            return
        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        states = torch.stack(states)
        next_states = torch.stack(next_states)
        actions = torch.tensor(actions).unsqueeze(1)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        dones = torch.tensor(dones, dtype=torch.float32)

        current_q = self.q_net(states).gather(1, actions).squeeze()
        next_q = self.target_net(next_states).max(1)[0]
        target_q = rewards + self.gamma * next_q * (1 - dones)

        loss = F.mse_loss(current_q, target_q.detach())
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())


# 6. Main pipeline
if __name__ == "__main__":

    print("MINI SCIENTIFIC DISCOVERY ENGINE – Excel Dataset\n")

    # Try to find the file with different names/extensions
    possible_files = ['Lipophilicity.xlsx', 'Lipophilicity.csv', 'lipophilicity.xlsx', 'lipophilicity.csv']
    filepath = None

    for f in possible_files:
        if os.path.exists(f):
            filepath = f
            break

    if filepath is None:
        print("Error: Could not find Lipophilicity file.")
        print("Please ensure you have either 'Lipophilicity.xlsx' or 'Lipophilicity.csv' in the current directory.")
        print("The file should contain at least two columns: SMILES and experimental logP values.")
        import sys
        sys.exit(1)

    # ---- Step 1: Load data ----
    try:
        graphs, targets = load_lipophilicity_excel(filepath, max_rows=2000)
    except Exception as e:
        print(f"Error loading data: {e}")
        import sys
        sys.exit(1)

    # Normalise logP values
    mean = targets.mean().item()
    std = targets.std().item()
    targets_norm = (targets - mean) / std
    print(f"logP statistics – Mean: {mean:.2f}, Std: {std:.2f}")

    # Train/validation split
    train_graphs, val_graphs, train_targets, val_targets = train_test_split(
        graphs, targets_norm, test_size=0.2, random_state=42
    )
    print(f"Training set: {len(train_graphs)} molecules")
    print(f"Validation set: {len(val_graphs)} molecules")

    # ---- Step 2: Train GNN ----

    print("STEP 2: Training GNN on lipophilicity data")


    gnn = GNNPredictor()
    optimizer_gnn = optim.Adam(gnn.parameters(), lr=0.001)
    best_val_loss = float('inf')

    for epoch in range(200):
        # Training
        gnn.train()
        optimizer_gnn.zero_grad()
        batch = Batch.from_data_list(train_graphs)
        pred = gnn(batch)
        loss = F.mse_loss(pred, train_targets)
        loss.backward()
        optimizer_gnn.step()

        # Validation
        gnn.eval()
        with torch.no_grad():
            val_batch = Batch.from_data_list(val_graphs)
            val_pred = gnn(val_batch)
            val_loss = F.mse_loss(val_pred, val_targets).item()

        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss

    print(f"✓ GNN training complete. Best validation loss: {best_val_loss:.4f}")

    # Step 3: DQN discovery

    print("STEP 3: DQN discovering optimal substituted benzenes")


    state_dim = HIDDEN_DIM
    n_actions = ACTION_SPACE * len(POSITIONS)   # 5*4 = 20
    agent = SimpleDQN(gnn, state_dim, n_actions, lr=0.001, target_norm=(mean, std))

    current_combo = [0, 0, 0, 0]   # start with benzene
    best_logp = -float('inf')
    best_combo = current_combo.copy()

    for episode in range(200):
        s_emb = agent.get_state_embedding(current_combo)
        if s_emb is None:
            current_combo = [0, 0, 0, 0]
            continue

        a = agent.choose_action(s_emb)
        pos = a // len(FRAGMENTS)
        frag = a % len(FRAGMENTS)
        next_combo = current_combo.copy()
        next_combo[pos] = frag

        # Reward = predicted logP (denormalized)
        reward = agent.predict_logp(next_combo)
        if reward is None:
            reward = -5.0
            done = True
        else:
            done = False
            if reward > best_logp:
                best_logp = reward
                best_combo = next_combo.copy()

        s_next_emb = agent.get_state_embedding(next_combo) if not done else torch.zeros(state_dim)
        agent.store_transition(s_emb, a, reward, s_next_emb, done)
        agent.update()

        if not done:
            current_combo = next_combo
        else:
            current_combo = [0, 0, 0, 0]

        if episode % 20 == 0:
            agent.update_target()
            smi = build_molecule(current_combo)
            if smi:
                mol = Chem.MolFromSmiles(smi)
                rdkit_logp = Descriptors.MolLogP(mol) if mol else float('nan')
                pred_logp = agent.predict_logp(current_combo)
                print(f"Ep {episode:3d} | {smi:>25s} | Pred logP: {pred_logp:.2f} | RDKit logP: {rdkit_logp:.2f}")

    # Final results

    print("FINAL RESULTS")


    final_smi = build_molecule(best_combo)
    if final_smi:
        mol = Chem.MolFromSmiles(final_smi)
        rdkit_logp = Descriptors.MolLogP(mol) if mol else None

        print(f"\nBest discovered molecule:")
        print(f"   SMILES:        {final_smi}")
        print(f"   Substituents:  {[FRAGMENTS[i] if FRAGMENTS[i] else 'H' for i in best_combo]}")
        print(f"   Predicted logP: {best_logp:.2f}")
        if rdkit_logp:
            print(f"   RDKit logP:     {rdkit_logp:.2f}")

    print(f"\nTop 5 high-logP combinations explored (for comparison):")
    top_combos = [
        [3, 3, 3, 3],  # all Cl
        [1, 1, 1, 1],  # all CH3
        [0, 3, 3, 3],  # benzene with 3 Cl
        [1, 1, 3, 3],  # 2 CH3, 2 Cl
        best_combo
    ]

    for combo in top_combos:
        smi = build_molecule(combo)
        if smi:
            mol = Chem.MolFromSmiles(smi)
            rdk_logp = Descriptors.MolLogP(mol) if mol else float('nan')
            pred_logp = agent.predict_logp(combo)
            fragments = [FRAGMENTS[i] if FRAGMENTS[i] else 'H' for i in combo]
            print(f"   {smi:>25s} | Frags: {fragments} | Pred: {pred_logp:.2f} | RDKit: {rdk_logp:.2f}")

MINI SCIENTIFIC DISCOVERY ENGINE – Excel Dataset

Loading data from Lipophilicity.csv ...
File contains 4200 rows
Using: SMILES = 'smiles', logP = 'exp'
Loaded 2000 valid molecules (skipped 0 invalid).
logP statistics – Mean: 2.17, Std: 1.21
Training set: 1600 molecules
Validation set: 400 molecules
STEP 2: Training GNN on lipophilicity data
Epoch   0 | Train Loss: 1.0112 | Val Loss: 0.9505
Epoch  20 | Train Loss: 1.0026 | Val Loss: 0.9436
Epoch  40 | Train Loss: 0.9846 | Val Loss: 0.9283
Epoch  60 | Train Loss: 0.9512 | Val Loss: 0.9007
Epoch  80 | Train Loss: 0.9106 | Val Loss: 0.8759
Epoch 100 | Train Loss: 0.8802 | Val Loss: 0.8609
Epoch 120 | Train Loss: 0.8615 | Val Loss: 0.8519
Epoch 140 | Train Loss: 0.8486 | Val Loss: 0.8434
Epoch 160 | Train Loss: 0.8381 | Val Loss: 0.8362
Epoch 180 | Train Loss: 0.8269 | Val Loss: 0.8254
✓ GNN training complete. Best validation loss: 0.8195
STEP 3: DQN discovering optimal substituted benzenes
Ep   0 |                Clc1ccccc1 | Pred logP: 3

In [ ]:
pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.1 MB/s eta 0:00:00


In [ ]:
pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 12.7 MB/s eta 0:00:00
